# DSA and Problem-Solving Foundations

Maps to `design3.md` Phase 2.

This notebook is for building structural problem-solving ability, not for random LeetCode volume. Every section explains **why** a pattern works, connects it to real data-engineering work, and includes fully traced worked problems so you can internalize the reasoning, not just memorize the code.

Goal: given a problem, you should be able to identify the right data structure or algorithm, explain why it fits, implement it correctly, and state its time and space complexity without guessing.

Working rule:

- read the explanation
- trace the worked example by hand before running the code
- run the code and verify your trace
- modify the example to test edge cases
- write one short explanation in your own words before moving on

## Learning Goals

By the end of this notebook, you should be able to:

- analyze time and space complexity of any solution you write, including amortized costs
- choose the right data structure for a problem and justify the choice with complexity reasoning
- implement hash map, sliding window, stack/queue, binary search, graph traversal, topological sort, heap, and DP solutions from scratch
- trace each algorithm step by step and explain what happens at every decision point
- connect every pattern to real platform/pipeline/trading-system work
- recognize problem shapes: "this is a graph problem", "this needs a sliding window", "this has overlapping subproblems"

## Sections

1. Complexity Analysis (Big-O)
2. Hash Maps
3. Sliding Window
4. Stacks and Queues
5. Binary Search
6. Graphs: BFS and DFS
7. Topological Sort
8. Heaps / Priority Queues
9. Dynamic Programming
10. Data Structure Selection Guide
11. Problem Bank
12. Mini Lab
13. Exit Criteria
14. References

## 1. Complexity Analysis: Big-O, Amortized Cost, and Space

Before solving any problem, you need the language to describe its cost.

**Big-O: worst-case growth rate**

| Complexity | Name | Example |
| --- | --- | --- |
| O(1) | Constant | Dict lookup, list append (amortized) |
| O(log n) | Logarithmic | Binary search, heap insert |
| O(n) | Linear | Single scan, list search |
| O(n log n) | Linearithmic | Timsort (Python's default sort) |
| O(n²) | Quadratic | Nested loops, naive duplicate check |
| O(2ⁿ) | Exponential | Recursive Fibonacci (naive) |
| O(n!) | Factorial | Brute-force permutation |

**Amortized cost**: the average cost over a sequence of operations.  
`list.append` is O(1) amortized — most appends are O(1), but occasionally Python doubles the backing array (O(n)). Over n appends the total is O(n), so each append costs O(1) on average.

**Space complexity**: measures peak memory usage, not just time.  
- A recursive DFS with depth n uses O(n) stack space
- A BFS with a wide graph can use O(n) queue space even if the graph itself is stored differently

**Common Python operation costs you must know:**

| Operation | Structure | Cost |
| --- | --- | --- |
| `d[key]` / `key in d` | `dict` | O(1) average |
| `x in s` | `set` | O(1) average |
| `x in lst` | `list` | O(n) |
| `lst.append(x)` | `list` | O(1) amortized |
| `lst.insert(0, x)` | `list` | O(n) |
| `lst.pop()` | `list` | O(1) |
| `lst.pop(0)` | `list` | O(n) — items shift |
| `sorted(lst)` | any | O(n log n) |
| `heapq.heappush` | `list` heap | O(log n) |
| `deque.appendleft` | `deque` | O(1) |

**Space trade-off principle**: many DSA problems trade space for time — pay O(n) space for a lookup structure to drop from O(n²) to O(n) time.

**Interview questions:**
- What is the difference between worst-case and amortized complexity?
- Why is `list.pop(0)` O(n) while `deque.popleft()` is O(1)?
- When does space complexity become the binding constraint?
- What is the time complexity of Python's `in` operator on a list vs. a set?


In [ ]:
import timeit
import random
from collections import deque

# Demonstrate O(n) vs O(1) membership: list vs set
n = 100_000
data = list(range(n))
data_set = set(data)
target = n - 1   # worst case: last element

list_time = timeit.timeit(lambda: target in data, number=1000)
set_time = timeit.timeit(lambda: target in data_set, number=1000)

# Demonstrate list.pop(0) O(n) vs deque.popleft() O(1)
lst = list(range(1000))
dq = deque(range(1000))

list_pop0_time = timeit.timeit(lambda: lst.insert(0, lst.pop(0)) if lst else None, number=10_000)
deque_pop_time = timeit.timeit(lambda: dq.appendleft(dq.popleft()) if dq else None, number=10_000)

# Demonstrate O(n²) vs O(n) duplicate detection
def has_duplicate_slow(lst):
    for i in range(len(lst)):
        for j in range(i + 1, len(lst)):
            if lst[i] == lst[j]:
                return True
    return False

def has_duplicate_fast(lst):
    return len(lst) != len(set(lst))

sample = list(range(500)) + [499]  # duplicate at end (worst case for slow)
slow_time = timeit.timeit(lambda: has_duplicate_slow(sample), number=100)
fast_time = timeit.timeit(lambda: has_duplicate_fast(sample), number=100)

{
    "list_membership_ms_per_1000": round(list_time * 1000, 2),
    "set_membership_ms_per_1000": round(set_time * 1000, 4),
    "speedup_set_vs_list": round(list_time / set_time, 0),
    "list_pop0_ms": round(list_pop0_time * 1000, 2),
    "deque_popleft_ms": round(deque_pop_time * 1000, 2),
    "duplicate_slow_ms": round(slow_time * 1000, 2),
    "duplicate_fast_ms": round(fast_time * 1000, 4),
}


## 2. Hash Maps

### How hash tables work

A hash table is an array of "buckets." To store a key-value pair:

1. Compute `hash(key)` -- a deterministic integer from the key
2. Map that integer to a bucket index: `index = hash(key) % num_buckets`
3. Store the key-value pair in that bucket

To look up a key, repeat steps 1-2, then check the bucket. If no collision, you find it immediately: O(1).

### Collision handling

Two different keys can hash to the same bucket index. This is a **collision**. Python's dict uses **open addressing** with probing: if the target slot is occupied, it probes subsequent slots using a perturbation scheme until it finds an empty one.

Average case: O(1) lookup, insert, delete. This holds as long as the hash table is not too full (Python keeps the load factor below ~2/3 by resizing).

Worst case: O(n) if every key collides into the same bucket chain. This is rare with Python's built-in hash but can happen with adversarial input. Python 3.12+ uses SipHash to resist hash-flooding DoS attacks.

### Why dict and set give O(1) average lookup

Because hash computation is O(1) for typical keys (strings, ints, tuples of those), and the bucket lookup is O(1) when collisions are rare. The dict/set automatically resizes when the load factor gets too high, keeping collisions rare.

### When hash tables degrade

- **Too many collisions**: bad hash function, all keys mapping to same bucket
- **Large keys**: if the hash function itself is expensive (e.g., hashing a 10MB string), O(1) hides a large constant
- **Memory pressure**: hash tables use more memory than arrays because they need spare capacity for low load factor

### Industry use cases

- **Symbol lookups**: `symbol_to_metadata: dict[str, Metadata]` -- constant-time routing of incoming trades to their config
- **Dedup sets**: `seen_ids: set[str]` -- O(1) check whether a message ID has already been processed (exactly-once semantics in Kafka consumers)
- **Aggregation buckets**: `defaultdict(list)` grouping records by key before batch writes
- **Dispatch tables**: `handler_map: dict[str, Callable]` -- map event types to handler functions instead of long if/elif chains

### Worked Problem: Two Sum (LC 1)

**Problem**: Given an array of integers `nums` and an integer `target`, return the indices of the two numbers that add up to `target`. Each input has exactly one solution.

**Thought process**:

1. Brute force: check every pair -- O(n^2). Too slow for large inputs.
2. Key insight: for each number `x`, we need `target - x`. This is a **lookup** problem. Hash maps give O(1) lookup.
3. Strategy: iterate once. For each element, check if `target - current` is already in the map. If yes, return both indices. If no, store `current -> index` in the map.
4. Time: O(n) -- single pass, each lookup is O(1). Space: O(n) -- storing up to n entries.
5. Edge cases: duplicate values (handled because we check before inserting), negative numbers (hash maps handle them fine).

### Worked Problem: Group Anagrams (LC 49)

**Problem**: Given a list of strings, group the anagrams together.

**Thought process**:

1. Two strings are anagrams if they contain the same characters with the same frequencies.
2. Key insight: we need a **canonical form** that is identical for all anagrams. Sorting the characters works: `"eat"` and `"tea"` both become `"aet"`.
3. Strategy: use sorted string as dict key, collect all original strings under that key.
4. Time: O(n * k log k) where n is the number of strings and k is max string length (sorting each string). Space: O(n * k).
5. Alternative: instead of sorting, use a tuple of character counts as the key -- O(n * k) time but more complex code.

In [ ]:
"""Hash Maps: Two Sum and Group Anagrams."""

from collections import defaultdict


# --- Two Sum ---
def two_sum(nums: list[int], target: int) -> list[int]:
    """O(n) time, O(n) space."""
    seen: dict[int, int] = {}  # value -> index
    for i, num in enumerate(nums):
        complement = target - num
        if complement in seen:
            return [seen[complement], i]
        seen[num] = i
    return []  # problem guarantees a solution exists


# Step-by-step trace for nums=[2, 7, 11, 15], target=9:
# i=0, num=2: complement=7, not in seen. seen={2:0}
# i=1, num=7: complement=2, 2 IS in seen at index 0. Return [0, 1].
print("Two Sum [2,7,11,15] target=9:", two_sum([2, 7, 11, 15], 9))
print("Two Sum [3,2,4] target=6:", two_sum([3, 2, 4], 6))
print("Two Sum [3,3] target=6:", two_sum([3, 3], 6))  # duplicate values


# --- Group Anagrams ---
def group_anagrams(strs: list[str]) -> list[list[str]]:
    """O(n * k log k) time, O(n * k) space."""
    groups: dict[str, list[str]] = defaultdict(list)
    for s in strs:
        key = "".join(sorted(s))  # canonical form
        groups[key].append(s)
    return list(groups.values())


# Trace for ["eat", "tea", "tan", "ate", "nat", "bat"]:
# "eat" -> sorted="aet" -> groups={"aet": ["eat"]}
# "tea" -> sorted="aet" -> groups={"aet": ["eat", "tea"]}
# "tan" -> sorted="ant" -> groups={"aet": ["eat", "tea"], "ant": ["tan"]}
# "ate" -> sorted="aet" -> groups={"aet": ["eat", "tea", "ate"], "ant": ["tan"]}
# "nat" -> sorted="ant" -> groups={"aet": ["eat", "tea", "ate"], "ant": ["tan", "nat"]}
# "bat" -> sorted="abt" -> groups={..., "abt": ["bat"]}
result = group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"])
print("\nGroup Anagrams:", result)


# --- Industry example: dedup with a set ---
def deduplicate_messages(messages: list[dict]) -> list[dict]:
    """Kafka consumer dedup using message IDs. O(n) time, O(n) space."""
    seen_ids: set[str] = set()
    unique: list[dict] = []
    for msg in messages:
        if msg["id"] not in seen_ids:
            seen_ids.add(msg["id"])
            unique.append(msg)
    return unique


messages = [
    {"id": "abc-001", "symbol": "EURUSD", "price": 1.0845},
    {"id": "abc-002", "symbol": "USDJPY", "price": 145.12},
    {"id": "abc-001", "symbol": "EURUSD", "price": 1.0845},  # duplicate
    {"id": "abc-003", "symbol": "GBPUSD", "price": 1.2641},
]
print(f"\nBefore dedup: {len(messages)} messages")
print(f"After dedup: {len(deduplicate_messages(messages))} messages")

# Try next:
# 1. Implement group_anagrams using a character-count tuple as the key instead of sorting.
# 2. Build a dispatch table that maps event types to handler functions.
# 3. What happens if you use a list as a dict key? Why?

## 3. Sliding Window

### The pattern

A sliding window maintains a **contiguous** subarray or substring, expanding and shrinking based on conditions. Instead of re-examining every possible subarray (O(n^2) or worse), you slide the window across the data in a single pass: O(n).

The general template:

1. Initialize two pointers: `left = 0`, `right` iterates forward
2. **Expand**: move `right` forward, adding the new element to your window state
3. **Shrink**: while the window violates a constraint, move `left` forward, removing elements from window state
4. **Record**: update the best answer after each adjustment

### When to use sliding window

The problem involves a **contiguous** subarray or substring and asks for:
- longest/shortest subarray satisfying a condition
- maximum/minimum sum of a fixed-size window
- subarray with at most K distinct elements

If the constraint is not about contiguous elements, sliding window does not apply.

### Industry use cases

- **Rolling statistics**: computing rolling VWAP, rolling average, or rolling max over the last N ticks in a trading system
- **Anomaly detection**: "alert if more than K errors occur in any 5-minute window" -- this is a fixed-size sliding window over timestamped events
- **Rate limiting**: token bucket / sliding window rate limiters count requests in a moving time window
- **Kafka windowed aggregation**: tumbling and sliding windows over streams of events -- the exact pattern from the JanAndFeb consumer

### Worked Problem: Longest Substring Without Repeating Characters (LC 3)

**Problem**: Given a string, find the length of the longest substring without repeating characters.

**Thought process**:

1. Brute force: check every substring for uniqueness -- O(n^3). Way too slow.
2. This asks for the **longest contiguous** substring meeting a condition (no repeats). That is a classic sliding window signal.
3. Window state: a dict mapping each character to its latest index. When we see a repeat, shrink the window by moving `left` past the previous occurrence.
4. Time: O(n) -- each character is visited at most twice (once by `right`, once by `left`). Space: O(min(n, alphabet_size)).
5. Edge cases: empty string (return 0), all identical characters (return 1), all unique characters (return n).

In [ ]:
"""Sliding Window: Longest Substring Without Repeating Characters."""


def length_of_longest_substring(s: str) -> int:
    """O(n) time, O(min(n, alphabet)) space."""
    seen: dict[str, int] = {}  # char -> most recent index
    left = 0
    best = 0

    for right, ch in enumerate(s):
        if ch in seen and seen[ch] >= left:
            # Shrink: move left past the previous occurrence of ch
            left = seen[ch] + 1
        seen[ch] = right
        best = max(best, right - left + 1)

    return best


# Step-by-step trace for s = "abcabcbb":
# right=0, ch='a': seen={'a':0}, window="a",    best=1
# right=1, ch='b': seen={'a':0,'b':1}, window="ab",   best=2
# right=2, ch='c': seen={'a':0,'b':1,'c':2}, window="abc",  best=3
# right=3, ch='a': 'a' in seen at 0, left=1. seen={'a':3,'b':1,'c':2}, window="bca", best=3
# right=4, ch='b': 'b' in seen at 1, left=2. seen={'a':3,'b':4,'c':2}, window="cab", best=3
# right=5, ch='c': 'c' in seen at 2, left=3. seen={'a':3,'b':4,'c':5}, window="abc", best=3
# right=6, ch='b': 'b' in seen at 4, left=5. seen={'a':3,'b':6,'c':5}, window="cb",  best=3
# right=7, ch='b': 'b' in seen at 6, left=7. seen={'a':3,'b':7,'c':5}, window="b",   best=3
# Answer: 3 ("abc")

print("'abcabcbb':", length_of_longest_substring("abcabcbb"))   # 3
print("'bbbbb':", length_of_longest_substring("bbbbb"))         # 1
print("'pwwkew':", length_of_longest_substring("pwwkew"))       # 3
print("'':", length_of_longest_substring(""))                   # 0
print("'abcdef':", length_of_longest_substring("abcdef"))       # 6


# --- Industry example: rolling average over fixed window ---
def rolling_average(prices: list[float], window_size: int) -> list[float]:
    """Fixed-size sliding window. O(n) time, O(1) extra space."""
    if len(prices) < window_size:
        return []
    result: list[float] = []
    window_sum = sum(prices[:window_size])
    result.append(window_sum / window_size)

    for i in range(window_size, len(prices)):
        window_sum += prices[i] - prices[i - window_size]  # slide: add new, drop old
        result.append(window_sum / window_size)

    return result


tick_prices = [1.0845, 1.0847, 1.0846, 1.0850, 1.0848, 1.0852, 1.0849]
print(f"\nPrices: {tick_prices}")
print(f"3-tick rolling avg: {[round(x, 4) for x in rolling_average(tick_prices, 3)]}")

# Try next:
# 1. Modify to also track the max element in each window (hint: use a deque as monotonic queue).
# 2. Implement "max sum subarray of size K" using a fixed-size sliding window.
# 3. Solve "minimum window substring" -- variable-size window with a frequency constraint.

## 4. Stacks and Queues

### Stack (LIFO -- Last In, First Out)

A stack is like a pile of plates: you add to the top and remove from the top. The last element pushed is the first one popped.

Core operations: `push` (append to top), `pop` (remove from top), `peek` (look at top without removing). All O(1).

In Python, use a plain `list` as a stack: `append()` to push, `pop()` to pop from the end. Both are O(1).

**When to use a stack**:
- **Matching/nesting problems**: parentheses validation, HTML tag matching, expression evaluation
- **Undo operations**: each action is pushed; undo pops the most recent
- **Function call simulation**: DFS uses a stack (the call stack in recursion, or an explicit stack iteratively)
- **Monotonic stack**: finding next greater/smaller element in O(n)

### Queue (FIFO -- First In, First Out)

A queue is like a line at a counter: first to arrive is first to be served.

Core operations: `enqueue` (add to back), `dequeue` (remove from front). Both should be O(1).

In Python, use `collections.deque`: `append()` to enqueue, `popleft()` to dequeue. Both are O(1). Do NOT use `list.pop(0)` -- that is O(n) because it shifts every element.

**When to use a queue**:
- **BFS traversal**: explore nodes level by level
- **Task scheduling**: process jobs in arrival order (Celery, Airflow task queues)
- **Producer-consumer buffers**: Kafka partitions are essentially queues
- **Rate limiting**: track request timestamps in a deque, evict old ones from the left

### `collections.deque`

A deque (double-ended queue) supports O(1) append/pop from both ends. It is the correct choice for:
- queues (append right, pop left)
- sliding window maximums (monotonic deque)
- bounded buffers (`deque(maxlen=N)` automatically drops oldest when full)

### Industry use cases

- **Expression parsing**: evaluating or validating configuration expressions, SQL-like query parsing
- **Task queues**: Celery workers pull tasks from a FIFO queue; Airflow schedules tasks from a priority queue
- **BFS in dependency resolution**: resolving package or task dependencies level by level
- **Backpressure buffers**: bounded deque as a ring buffer for recent events; when full, oldest events are dropped

### Worked Problem: Valid Parentheses (LC 20)

**Problem**: Given a string containing `()[]{}`, determine if the input string is valid. Valid means every open bracket is closed by the same type in the correct order.

**Thought process**:

1. This is a **nesting/matching** problem. Nesting = stack.
2. Strategy: push open brackets onto the stack. When we see a close bracket, pop the stack and check if it matches. If the stack is empty when we need to pop, or the final stack is not empty, the string is invalid.
3. Time: O(n) -- single pass. Space: O(n) -- stack could hold all characters in worst case.
4. Edge cases: empty string (valid), odd length (immediately invalid), only open brackets (invalid -- stack not empty at end).

In [ ]:
"""Stacks and Queues: Valid Parentheses and industry examples."""

from collections import deque


# --- Valid Parentheses ---
def is_valid(s: str) -> bool:
    """O(n) time, O(n) space."""
    stack: list[str] = []
    matching = {")": "(", "]": "[", "}": "{"}

    for ch in s:
        if ch in matching:
            # Close bracket: check if stack top matches
            if not stack or stack[-1] != matching[ch]:
                return False
            stack.pop()
        else:
            # Open bracket: push
            stack.append(ch)

    return len(stack) == 0  # valid only if all brackets matched


# Trace for s = "({[]})":
# ch='(': push. stack=['(']
# ch='{': push. stack=['(', '{']
# ch='[': push. stack=['(', '{', '[']
# ch=']': matches '[', pop. stack=['(', '{']
# ch='}': matches '{', pop. stack=['(']
# ch=')': matches '(', pop. stack=[]
# Stack empty -> True

print("'({[]})':", is_valid("({[]})"))   # True
print("'([)]':", is_valid("([)]"))       # False
print("'':", is_valid(""))               # True
print("'(((':", is_valid("((("))         # False
print("')))':", is_valid(")))"))         # False


# --- Industry example: bounded event buffer ---
# A deque with maxlen acts as a ring buffer. When full, oldest events drop automatically.
event_buffer: deque[dict] = deque(maxlen=3)
for i, event in enumerate(["trade-A", "trade-B", "trade-C", "trade-D", "trade-E"]):
    event_buffer.append({"seq": i, "event": event})
    print(f"  After adding {event}: {[e['event'] for e in event_buffer]}")

print(f"\nBuffer holds last 3: {[e['event'] for e in event_buffer]}")


# --- Industry example: simple task queue ---
task_queue: deque[str] = deque()
task_queue.append("ingest-batch-001")
task_queue.append("ingest-batch-002")
task_queue.append("ingest-batch-003")

while task_queue:
    task = task_queue.popleft()  # FIFO: process oldest first
    print(f"Processing: {task}")

# Try next:
# 1. Implement a min-stack that supports push, pop, top, and getMin all in O(1).
# 2. Implement a simple expression evaluator using two stacks (one for values, one for operators).
# 3. Why is deque better than list for queue operations? Time it.

## 5. Binary Search

### The pattern

Binary search eliminates **half the search space** at each step. It works on any monotonic structure -- most commonly a sorted array.

The invariant: at every step, the answer is guaranteed to be in `[left, right]`. You compute `mid`, check a condition, and eliminate the half that cannot contain the answer.

### Variants

1. **Exact match**: find `target` in a sorted array. Return index or -1.
2. **Lower bound** (`bisect_left`): find the leftmost position where `target` could be inserted to maintain order. Useful for "first occurrence" or "first element >= target."
3. **Upper bound** (`bisect_right`): find the rightmost insertion point. Useful for "last element <= target."
4. **Rotated sorted array**: the array was sorted then rotated. One half is always sorted; use that to decide which half to search.

### Common pitfall: off-by-one errors

Binary search bugs almost always come from boundary handling. Rules:

- `while left <= right` for exact match (closed interval `[left, right]`)
- `while left < right` for convergence to a boundary (half-open interval `[left, right)`)
- `mid = left + (right - left) // 2` to avoid overflow (relevant in other languages; habit is still good)

### Industry use cases

- **Searching sorted time-series data**: finding the first trade after a given timestamp in a sorted log. TimescaleDB uses B-tree indexes which are binary search internally.
- **Finding insertion points**: `bisect.insort` to maintain sorted order in a streaming buffer
- **Threshold finding**: binary search on the answer space -- "what is the minimum batch size that processes all records within the SLA?"
- **Python's `bisect` module**: `bisect_left`, `bisect_right`, `insort` -- use these in production instead of hand-rolling

### Worked Problem: Binary Search (LC 704)

**Problem**: Given a sorted array and target, return the index of target or -1.

**Thought process**: Textbook binary search. O(log n) time, O(1) space.

### Worked Problem: Search in Rotated Sorted Array (LC 33)

**Problem**: An array sorted in ascending order was rotated at some pivot. Given a target, return its index or -1. No duplicates.

**Thought process**:

1. The array has two sorted halves. Example: `[4,5,6,7,0,1,2]` -- `[4,5,6,7]` and `[0,1,2]`.
2. At each step, at least one of the two halves `[left..mid]` or `[mid..right]` is sorted.
3. Check which half is sorted, then check if target falls within that sorted range. If yes, search that half. If no, search the other half.
4. Time: O(log n). Space: O(1).
5. Edge cases: single element, target not present, array not rotated (just sorted).

In [ ]:
"""Binary Search: standard and rotated array."""

import bisect


# --- Standard Binary Search ---
def binary_search(nums: list[int], target: int) -> int:
    """O(log n) time, O(1) space."""
    left, right = 0, len(nums) - 1
    while left <= right:
        mid = left + (right - left) // 2
        if nums[mid] == target:
            return mid
        elif nums[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1


# Trace for nums=[-1,0,3,5,9,12], target=9:
# left=0, right=5, mid=2: nums[2]=3 < 9 -> left=3
# left=3, right=5, mid=4: nums[4]=9 == 9 -> return 4
print("Binary search [−1,0,3,5,9,12] target=9:", binary_search([-1, 0, 3, 5, 9, 12], 9))
print("Binary search [−1,0,3,5,9,12] target=2:", binary_search([-1, 0, 3, 5, 9, 12], 2))


# --- Search in Rotated Sorted Array ---
def search_rotated(nums: list[int], target: int) -> int:
    """O(log n) time, O(1) space."""
    left, right = 0, len(nums) - 1

    while left <= right:
        mid = left + (right - left) // 2
        if nums[mid] == target:
            return mid

        # Determine which half is sorted
        if nums[left] <= nums[mid]:
            # Left half [left..mid] is sorted
            if nums[left] <= target < nums[mid]:
                right = mid - 1  # target is in sorted left half
            else:
                left = mid + 1   # target is in right half
        else:
            # Right half [mid..right] is sorted
            if nums[mid] < target <= nums[right]:
                left = mid + 1   # target is in sorted right half
            else:
                right = mid - 1  # target is in left half

    return -1


# Trace for nums=[4,5,6,7,0,1,2], target=0:
# left=0, right=6, mid=3: nums[3]=7 != 0
#   nums[0]=4 <= nums[3]=7 -> left half sorted
#   4 <= 0 < 7? No -> search right: left=4
# left=4, right=6, mid=5: nums[5]=1 != 0
#   nums[4]=0 <= nums[5]=1 -> left half sorted
#   0 <= 0 < 1? Yes -> search left: right=4
# left=4, right=4, mid=4: nums[4]=0 == 0 -> return 4

print("\nRotated search [4,5,6,7,0,1,2] target=0:", search_rotated([4, 5, 6, 7, 0, 1, 2], 0))
print("Rotated search [4,5,6,7,0,1,2] target=3:", search_rotated([4, 5, 6, 7, 0, 1, 2], 3))
print("Rotated search [1] target=1:", search_rotated([1], 1))


# --- Industry example: bisect for time-series lookup ---
# Find the first trade at or after a given timestamp
timestamps = [1000, 1050, 1100, 1150, 1200, 1250, 1300]
query_ts = 1120

# bisect_left: first index where query_ts could be inserted (first element >= query_ts)
idx = bisect.bisect_left(timestamps, query_ts)
print(f"\nTimestamps: {timestamps}")
print(f"First trade at or after {query_ts}: index={idx}, ts={timestamps[idx] if idx < len(timestamps) else 'N/A'}")

# bisect_right: first index after query_ts (first element > query_ts)
idx_right = bisect.bisect_right(timestamps, 1100)
print(f"Trades after 1100: {timestamps[idx_right:]}")

# Try next:
# 1. Implement lower_bound (find first element >= target) without using bisect.
# 2. Binary search on answer: find minimum speed to finish all tasks within T hours.
# 3. Handle the rotated array case where duplicates are allowed (LC 81).

## 6. Graphs: BFS and DFS

### Graph representation

A graph is a set of **nodes** (vertices) connected by **edges**. Two common representations:

**Adjacency list** (most common in practice): a dict mapping each node to its list of neighbors. Space: O(V + E). Lookup of all neighbors: O(degree). Best for sparse graphs, which is most real-world graphs.

```python
graph = {
    "A": ["B", "C"],
    "B": ["A", "D"],
    "C": ["A"],
    "D": ["B"],
}
```

**Adjacency matrix**: a 2D array where `matrix[i][j] = 1` if there is an edge from i to j. Space: O(V^2). Edge existence check: O(1). Best for dense graphs or when you need fast edge-existence queries. Rare in data engineering.

### BFS (Breadth-First Search)

BFS explores the graph **level by level** using a queue. It visits all neighbors of the current node before moving to the next level.

Algorithm:
1. Start from a source node, add it to a queue, mark as visited
2. While queue is not empty: dequeue a node, process it, enqueue all unvisited neighbors
3. Each node is visited exactly once: O(V + E) time, O(V) space

**When to use BFS**:
- Shortest path in an **unweighted** graph (BFS naturally finds the shortest path because it explores level by level)
- Level-order traversal (binary tree level order)
- Finding all nodes within K hops

### DFS (Depth-First Search)

DFS explores as **deep** as possible along each branch before backtracking. Uses a stack (explicit or via recursion).

Algorithm:
1. Start from a source node, push onto stack, mark as visited
2. While stack is not empty: pop a node, process it, push all unvisited neighbors
3. O(V + E) time, O(V) space

**When to use DFS**:
- Exhaustive search / backtracking (word search, permutations, combinations)
- Cycle detection
- Connected components (start DFS from each unvisited node)
- Topological sort (DFS variant)

### BFS vs DFS decision

| Criterion | BFS | DFS |
|-----------|-----|-----|
| Shortest path (unweighted) | Yes -- guaranteed | No |
| Memory on wide graphs | High (stores entire level) | Lower (stores one path) |
| Memory on deep graphs | Lower | High (stack depth) |
| Exhaustive search | Works but wasteful | Natural fit |
| Level-order processing | Natural fit | Not natural |

### Industry use cases

- **Dependency graphs**: Airflow DAGs represent task dependencies. BFS/DFS determine execution order and detect cycles.
- **Network topology**: tracing connectivity between services, finding unreachable nodes
- **Data lineage**: tracing how a dataset was produced through a pipeline graph
- **Grid problems**: many data problems map to grids (geographic data, image processing) where cells are nodes and adjacency is 4-directional

### Worked Problem: Number of Islands (LC 200)

**Problem**: Given a 2D grid of '1' (land) and '0' (water), count the number of islands. An island is connected land cells (horizontally/vertically adjacent).

**Thought process**:

1. This is a **connected components** problem on a grid. Each island is a connected component.
2. Strategy: iterate through every cell. When we find an unvisited '1', start a DFS/BFS to mark all connected land cells as visited. Increment the island count.
3. Time: O(rows * cols) -- each cell visited at most once. Space: O(rows * cols) worst case for the visited set (or modify grid in-place).
4. DFS is slightly more natural here because the grid is bounded and recursion depth is manageable.
5. Edge cases: empty grid, all water, all land, single cell.

In [ ]:
"""Graphs: BFS, DFS, and Number of Islands."""

from collections import deque


# --- BFS template ---
def bfs(graph: dict[str, list[str]], start: str) -> list[str]:
    """O(V + E) time, O(V) space."""
    visited = {start}
    queue = deque([start])
    order: list[str] = []
    while queue:
        node = queue.popleft()
        order.append(node)
        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
    return order


# --- DFS template (iterative) ---
def dfs(graph: dict[str, list[str]], start: str) -> list[str]:
    """O(V + E) time, O(V) space."""
    visited: set[str] = set()
    stack = [start]
    order: list[str] = []
    while stack:
        node = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        order.append(node)
        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                stack.append(neighbor)
    return order


# Demo graph:
#   A -- B -- D
#   |         |
#   C -- E    F
graph = {
    "A": ["B", "C"],
    "B": ["A", "D"],
    "C": ["A", "E"],
    "D": ["B", "F"],
    "E": ["C"],
    "F": ["D"],
}

print("BFS from A:", bfs(graph, "A"))  # level order: A, B, C, D, E, F
print("DFS from A:", dfs(graph, "A"))  # deep-first: order depends on stack


# --- Number of Islands ---
def num_islands(grid: list[list[str]]) -> int:
    """O(rows * cols) time, O(rows * cols) space."""
    if not grid:
        return 0

    rows, cols = len(grid), len(grid[0])
    visited: set[tuple[int, int]] = set()
    count = 0

    def dfs_island(r: int, c: int) -> None:
        """DFS to mark all cells of an island as visited."""
        stack = [(r, c)]
        while stack:
            cr, cc = stack.pop()
            for dr, dc in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                nr, nc = cr + dr, cc + dc
                if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in visited and grid[nr][nc] == "1":
                    visited.add((nr, nc))
                    stack.append((nr, nc))

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == "1" and (r, c) not in visited:
                visited.add((r, c))
                dfs_island(r, c)
                count += 1

    return count


# Step-by-step trace for this grid:
# 1 1 0 0 0
# 1 1 0 0 0
# 0 0 1 0 0
# 0 0 0 1 1
#
# Start at (0,0)='1': DFS marks (0,0),(0,1),(1,0),(1,1). Island #1.
# Skip (0,2),(0,3),(0,4),(1,2),(1,3),(1,4) -- water or visited.
# (2,2)='1': DFS marks (2,2). Island #2.
# (3,3)='1': DFS marks (3,3),(3,4). Island #3.
# Answer: 3

test_grid = [
    ["1", "1", "0", "0", "0"],
    ["1", "1", "0", "0", "0"],
    ["0", "0", "1", "0", "0"],
    ["0", "0", "0", "1", "1"],
]
print(f"\nNumber of islands: {num_islands(test_grid)}")  # 3

# Edge cases
print(f"Empty grid: {num_islands([])}")          # 0
print(f"All water: {num_islands([['0']])}")      # 0
print(f"All land: {num_islands([['1']])}")       # 1

# Try next:
# 1. Implement BFS version of num_islands and compare the traversal order.
# 2. Modify to also return the size of each island.
# 3. Solve "max area of island" (LC 695) using the same pattern.

## 7. Topological Sort

### What it is

A topological sort produces a **linear ordering** of nodes in a directed acyclic graph (DAG) such that for every directed edge `u -> v`, node `u` appears before node `v` in the ordering.

If the graph has a cycle, no valid topological ordering exists. This is how you detect cycles in dependency graphs.

### Why this is your bread and butter

As a data/platform engineer, topological sort is **everywhere**:

- **Airflow DAGs**: the scheduler determines task execution order using topological sort. Tasks with no unmet dependencies run first, then their downstream tasks become eligible.
- **Build systems**: Make, Bazel, and dbt all use topological sort to determine build order.
- **Pipeline task ordering**: "run extract before transform, transform before load" -- that is a topological constraint.
- **Database migration ordering**: migration scripts often have dependencies; topological sort determines safe execution order.
- **Package dependency resolution**: pip and conda resolve install order using dependency graphs.

### Kahn's Algorithm (BFS-based)

This is the most intuitive approach and maps directly to how Airflow schedules tasks:

1. Compute the **in-degree** (number of incoming edges) for every node
2. Add all nodes with in-degree 0 to a queue -- these have no dependencies
3. While queue is not empty:
   - Dequeue a node, add it to the result
   - For each neighbor: decrement its in-degree. If in-degree becomes 0, enqueue it
4. If the result has fewer nodes than the graph, there is a **cycle**

Time: O(V + E). Space: O(V + E).

Think of it this way: in-degree 0 = "all dependencies met." Just like Airflow marking a task as "ready to run" when all upstream tasks have succeeded.

### DFS-based approach

1. Run DFS from each unvisited node
2. After finishing all neighbors of a node (post-order), push it onto a stack
3. The stack (reversed post-order) is the topological sort
4. Detect cycles by tracking nodes currently in the recursion stack

### Cycle detection

- **Kahn's**: if the result contains fewer than V nodes, there is a cycle (some nodes never reached in-degree 0)
- **DFS**: if during traversal you encounter a node that is currently in the recursion stack (being processed, not yet finished), there is a cycle

### Worked Problem: Course Schedule (LC 207)

**Problem**: There are `numCourses` courses labeled 0 to numCourses-1. You are given prerequisite pairs `[a, b]` meaning you must take course `b` before course `a`. Return whether it is possible to finish all courses (i.e., is the prerequisite graph a DAG?).

**Thought process**:

1. This is explicitly a **cycle detection** problem on a directed graph. If there is a cycle in prerequisites, you cannot complete all courses.
2. Model as a graph: each course is a node, each prerequisite `[a, b]` creates edge `b -> a`.
3. Run Kahn's algorithm. If the topological sort includes all nodes, return True. Otherwise, there is a cycle: return False.
4. Time: O(V + E). Space: O(V + E).
5. Edge cases: no prerequisites (trivially possible), single course, self-loop (impossible).

In [ ]:
"""Topological Sort: Kahn's algorithm, DFS-based, and Course Schedule."""

from collections import deque


# --- Kahn's Algorithm (BFS-based topological sort) ---
def topological_sort_kahn(num_nodes: int, edges: list[tuple[int, int]]) -> list[int]:
    """
    Kahn's algorithm. Returns topological order or empty list if cycle exists.
    O(V + E) time, O(V + E) space.
    """
    graph: dict[int, list[int]] = {i: [] for i in range(num_nodes)}
    indegree = [0] * num_nodes

    for src, dst in edges:
        graph[src].append(dst)
        indegree[dst] += 1

    # Start with all nodes that have no dependencies
    queue = deque([i for i in range(num_nodes) if indegree[i] == 0])
    order: list[int] = []

    while queue:
        node = queue.popleft()
        order.append(node)
        for neighbor in graph[node]:
            indegree[neighbor] -= 1
            if indegree[neighbor] == 0:
                queue.append(neighbor)

    return order if len(order) == num_nodes else []  # empty = cycle detected


# --- DFS-based topological sort ---
def topological_sort_dfs(num_nodes: int, edges: list[tuple[int, int]]) -> list[int]:
    """
    DFS-based. Returns topological order or empty list if cycle exists.
    O(V + E) time, O(V + E) space.
    """
    graph: dict[int, list[int]] = {i: [] for i in range(num_nodes)}
    for src, dst in edges:
        graph[src].append(dst)

    WHITE, GRAY, BLACK = 0, 1, 2  # unvisited, in-progress, finished
    color = [WHITE] * num_nodes
    order: list[int] = []
    has_cycle = False

    def dfs(node: int) -> None:
        nonlocal has_cycle
        if has_cycle:
            return
        color[node] = GRAY  # mark as in-progress
        for neighbor in graph[node]:
            if color[neighbor] == GRAY:
                has_cycle = True  # back edge = cycle
                return
            if color[neighbor] == WHITE:
                dfs(neighbor)
        color[node] = BLACK  # mark as finished
        order.append(node)   # post-order

    for node in range(num_nodes):
        if color[node] == WHITE:
            dfs(node)

    if has_cycle:
        return []
    return order[::-1]  # reverse post-order = topological order


# --- Airflow-style pipeline example ---
# Tasks: 0=extract, 1=validate, 2=transform, 3=enrich, 4=load, 5=notify
# Dependencies: extract->validate, validate->transform, validate->enrich,
#               transform->load, enrich->load, load->notify
task_names = {0: "extract", 1: "validate", 2: "transform", 3: "enrich", 4: "load", 5: "notify"}
pipeline_edges = [(0, 1), (1, 2), (1, 3), (2, 4), (3, 4), (4, 5)]

order = topological_sort_kahn(6, pipeline_edges)
print("Pipeline execution order (Kahn's):")
for i, node in enumerate(order):
    print(f"  Step {i + 1}: {task_names[node]}")

order_dfs = topological_sort_dfs(6, pipeline_edges)
print(f"\nDFS-based order: {[task_names[n] for n in order_dfs]}")


# --- Course Schedule (LC 207) ---
def can_finish(num_courses: int, prerequisites: list[list[int]]) -> bool:
    """O(V + E) time, O(V + E) space."""
    # Build graph: prerequisite [a, b] means b -> a (take b before a)
    graph: dict[int, list[int]] = {i: [] for i in range(num_courses)}
    indegree = [0] * num_courses

    for course, prereq in prerequisites:
        graph[prereq].append(course)
        indegree[course] += 1

    queue = deque([i for i in range(num_courses) if indegree[i] == 0])
    completed = 0

    while queue:
        node = queue.popleft()
        completed += 1
        for neighbor in graph[node]:
            indegree[neighbor] -= 1
            if indegree[neighbor] == 0:
                queue.append(neighbor)

    return completed == num_courses


# No cycle: can finish
print(f"\nCan finish 4 courses, prereqs [[1,0],[2,0],[3,1],[3,2]]: {can_finish(4, [[1,0],[2,0],[3,1],[3,2]])}")

# Cycle: 0->1->0
print(f"Can finish 2 courses, prereqs [[1,0],[0,1]]: {can_finish(2, [[1,0],[0,1]])}")

# No prerequisites
print(f"Can finish 3 courses, no prereqs: {can_finish(3, [])}")

# Try next:
# 1. Modify to return the actual course order (Course Schedule II, LC 210).
# 2. Add a "parallel execution" layer: group nodes by their topological level.
# 3. Model your actual Airflow DAG as edges and verify the execution order.

## 8. Heaps / Priority Queues

### What a heap is

A heap is a **complete binary tree** stored as an array where every parent is smaller (min-heap) or larger (max-heap) than its children. This gives you:

- **Get minimum**: O(1) -- just look at index 0
- **Insert**: O(log n) -- add at end, bubble up
- **Remove minimum**: O(log n) -- swap root with last, bubble down
- **Build heap from array**: O(n) -- not O(n log n); the math works out due to most nodes being near leaves

A heap is NOT fully sorted. It only guarantees the root is the min (or max). The rest is partially ordered.

### Python's `heapq` module

Python provides a **min-heap** via `heapq`:

- `heapq.heappush(heap, item)` -- insert in O(log n)
- `heapq.heappop(heap)` -- remove and return smallest in O(log n)
- `heapq.heapify(list)` -- convert list to heap in-place in O(n)
- `heapq.nlargest(k, iterable)` and `heapq.nsmallest(k, iterable)` -- top-K in O(n + k log n)

**Max-heap trick**: negate values. Push `-x`, pop gives `-x` which you negate back to `x`.

### When to use a heap

- **Top-K problems**: find K largest/smallest elements without sorting everything -- O(n log k) vs O(n log n)
- **Priority scheduling**: always process the highest-priority task next
- **Merge K sorted lists**: use a heap to track the current smallest across all lists
- **Streaming median**: use two heaps (max-heap for lower half, min-heap for upper half)

### Industry use cases

- **Priority task scheduling**: Airflow priority weights, Celery priority queues -- highest priority task dequeued first
- **Top-N outliers**: in a monitoring system, maintain a heap of the N largest latency values seen in the last hour
- **Streaming median for anomaly detection**: track the median price in real-time for anomaly alerts. Two-heap approach handles this in O(log n) per insert.
- **Merge K sorted partitions**: when reading from K Kafka partitions with ordered timestamps, a heap efficiently produces a globally sorted stream

### Worked Problem: Find Median from Data Stream (LC 295)

**Problem**: Design a data structure that supports `addNum(num)` and `findMedian()` for a stream of numbers.

**Thought process**:

1. Sorting after each insert: O(n log n) per add. Too slow for streaming.
2. Key insight: we only need the **middle** element(s). Split the stream into a lower half and upper half. The median is derived from the boundary between them.
3. Use two heaps:
   - `max_heap`: stores the smaller half (negate values for max-heap behavior)
   - `min_heap`: stores the larger half
   - Keep sizes balanced: max_heap can have at most 1 more element than min_heap
4. `addNum`: O(log n). `findMedian`: O(1).
5. Edge cases: single element, two elements, all same values.

In [ ]:
"""Heaps: priority queues, top-K, and streaming median."""

import heapq


# --- Basic heap operations ---
prices = [1.0850, 1.0845, 1.0848, 1.0842, 1.0847]
heapq.heapify(prices)  # O(n) -- converts list in-place to a min-heap
print(f"Heap (min at front): {prices}")
print(f"Smallest: {prices[0]}")  # O(1) peek
print(f"Pop smallest: {heapq.heappop(prices)}")  # O(log n)
heapq.heappush(prices, 1.0840)  # O(log n)
print(f"After push 1.0840: {prices}")


# --- Top-K using a min-heap of size K ---
def top_k_largest(nums: list[float], k: int) -> list[float]:
    """
    O(n log k) time, O(k) space.
    Maintain a min-heap of size k. The heap always contains the k largest seen so far.
    """
    heap: list[float] = []
    for num in nums:
        if len(heap) < k:
            heapq.heappush(heap, num)
        elif num > heap[0]:
            heapq.heapreplace(heap, num)  # pop smallest, push new
    return sorted(heap, reverse=True)


latencies = [12.5, 3.2, 45.1, 8.7, 99.3, 1.1, 67.4, 23.8, 15.6, 88.2]
print(f"\nLatencies: {latencies}")
print(f"Top 3 largest: {top_k_largest(latencies, 3)}")


# --- Streaming Median (two-heap approach) ---
class MedianFinder:
    """
    addNum: O(log n), findMedian: O(1).
    max_heap (negated) holds the smaller half, min_heap holds the larger half.
    """

    def __init__(self) -> None:
        self.max_heap: list[float] = []  # negated values for max-heap behavior
        self.min_heap: list[float] = []

    def add_num(self, num: float) -> None:
        # Always add to max_heap first (smaller half)
        heapq.heappush(self.max_heap, -num)

        # Ensure max_heap's largest <= min_heap's smallest
        if self.min_heap and (-self.max_heap[0]) > self.min_heap[0]:
            val = -heapq.heappop(self.max_heap)
            heapq.heappush(self.min_heap, val)

        # Balance sizes: max_heap can have at most 1 more element
        if len(self.max_heap) > len(self.min_heap) + 1:
            val = -heapq.heappop(self.max_heap)
            heapq.heappush(self.min_heap, val)
        elif len(self.min_heap) > len(self.max_heap):
            val = heapq.heappop(self.min_heap)
            heapq.heappush(self.max_heap, -val)

    def find_median(self) -> float:
        if len(self.max_heap) > len(self.min_heap):
            return -self.max_heap[0]
        return (-self.max_heap[0] + self.min_heap[0]) / 2.0


# Trace: adding [5, 2, 8, 1, 9]
mf = MedianFinder()
stream = [5, 2, 8, 1, 9]
print("\nStreaming median:")
for num in stream:
    mf.add_num(num)
    print(f"  Added {num}: median = {mf.find_median()}")
# After [5]:       median=5
# After [5,2]:     median=3.5
# After [5,2,8]:   median=5
# After [5,2,8,1]: median=3.5
# After [5,2,8,1,9]: median=5


# --- Industry example: priority task scheduler ---
tasks = [
    (3, "load-to-warehouse"),      # priority 3 (lower = higher priority)
    (1, "critical-alert-check"),   # priority 1
    (2, "transform-batch"),        # priority 2
    (1, "validate-schema"),        # priority 1
]
heapq.heapify(tasks)
print("\nTask execution order (priority scheduler):")
while tasks:
    priority, task_name = heapq.heappop(tasks)
    print(f"  Priority {priority}: {task_name}")

# Try next:
# 1. Implement merge_k_sorted_lists using a heap.
# 2. Find the K closest points to the origin using a max-heap of size K.
# 3. Why is heapify O(n) and not O(n log n)? Reason about it.

## 9. Dynamic Programming

### What DP actually is

Dynamic programming solves problems by breaking them into **overlapping subproblems** that share **optimal substructure**. Instead of recomputing the same subproblem many times (like naive recursion), DP computes each subproblem once and stores the result.

Two requirements for DP to apply:

1. **Overlapping subproblems**: the same smaller problem is solved multiple times. Example: Fibonacci -- `fib(5)` needs `fib(4)` and `fib(3)`, but `fib(4)` also needs `fib(3)`. Without caching, `fib(3)` is computed twice.
2. **Optimal substructure**: the optimal solution to the problem can be constructed from optimal solutions to its subproblems.

### The systematic approach

Every DP solution follows this framework:

1. **Define state**: what does `dp[i]` (or `dp[i][j]`) represent? This is the hardest step.
2. **Find recurrence**: how does `dp[i]` relate to smaller subproblems? This is the transition.
3. **Set base case**: what are the trivially solvable subproblems?
4. **Determine build order**: bottom-up (iterative) or top-down (recursive + memoization)
5. **Extract answer**: usually `dp[n]` or `dp[n-1]`

### Top-down (memoization) vs bottom-up (tabulation)

**Top-down**: write the recursive solution, add a cache (`@functools.cache` or a dict). Natural to write, follows the problem's recursive structure. Downside: recursion depth limit, function call overhead.

**Bottom-up**: fill a table iteratively from base cases up. No recursion stack risk, often slightly faster. Downside: you must determine the correct iteration order.

Both have the same time and space complexity. Bottom-up is generally preferred in production for the lack of stack risk.

### How to recognize DP problems

Signal words and patterns:

- "count the number of ways to..."
- "find the minimum/maximum cost to..."
- "is it possible to..." (with choices that affect future options)
- "longest/shortest subsequence" (not substring -- that might be sliding window)
- Choices at each step that affect future options, and the same substate is reached multiple ways

### Industry use cases

- **Resource allocation**: given N workers and K tasks with costs, minimize total cost (assignment problems)
- **Scheduling optimization**: minimize total pipeline latency given task dependencies and resource constraints
- **String matching**: edit distance for fuzzy matching of vendor names, product codes
- **Batch sizing**: minimize total processing cost given variable batch sizes and overhead costs

### Worked Problems

Three problems of increasing difficulty, each fully traced.

In [ ]:
"""Dynamic Programming: Climbing Stairs, Coin Change, House Robber."""

from functools import cache


# =============================================================================
# Problem 1: Climbing Stairs (LC 70)
# =============================================================================
# You can climb 1 or 2 steps. How many distinct ways to reach step n?
#
# State:   dp[i] = number of ways to reach step i
# Recurrence: dp[i] = dp[i-1] + dp[i-2]  (arrive from 1 step back OR 2 steps back)
# Base case: dp[0] = 1 (one way to stand at ground), dp[1] = 1
# Answer:  dp[n]

def climb_stairs(n: int) -> int:
    """O(n) time, O(1) space (optimized -- only need last 2 values)."""
    if n <= 2:
        return n
    prev2, prev1 = 1, 2
    for i in range(3, n + 1):
        curr = prev1 + prev2
        prev2, prev1 = prev1, curr
    return prev1


# Trace for n=5:
# dp[1]=1, dp[2]=2
# dp[3] = dp[2]+dp[1] = 2+1 = 3
# dp[4] = dp[3]+dp[2] = 3+2 = 5
# dp[5] = dp[4]+dp[3] = 5+3 = 8
print("Climbing stairs:")
for n in range(1, 8):
    print(f"  n={n}: {climb_stairs(n)} ways")


# =============================================================================
# Problem 2: Coin Change (LC 322)
# =============================================================================
# Given coins of certain denominations, find minimum coins to make amount.
# Return -1 if impossible.
#
# State:   dp[i] = minimum coins needed to make amount i
# Recurrence: dp[i] = min(dp[i - coin] + 1) for each coin where i - coin >= 0
# Base case: dp[0] = 0 (zero coins to make amount 0)
# Answer:  dp[amount]

def coin_change(coins: list[int], amount: int) -> int:
    """O(amount * len(coins)) time, O(amount) space."""
    dp = [float("inf")] * (amount + 1)
    dp[0] = 0

    for i in range(1, amount + 1):
        for coin in coins:
            if coin <= i and dp[i - coin] + 1 < dp[i]:
                dp[i] = dp[i - coin] + 1

    return dp[amount] if dp[amount] != float("inf") else -1


# Trace for coins=[1,3,4], amount=6:
# dp[0]=0
# dp[1]: try coin 1 -> dp[0]+1=1. dp[1]=1
# dp[2]: try coin 1 -> dp[1]+1=2. dp[2]=2
# dp[3]: try coin 1 -> dp[2]+1=3. try coin 3 -> dp[0]+1=1. dp[3]=1
# dp[4]: try coin 1 -> dp[3]+1=2. try coin 3 -> dp[1]+1=2. try coin 4 -> dp[0]+1=1. dp[4]=1
# dp[5]: try coin 1 -> dp[4]+1=2. try coin 3 -> dp[2]+1=3. try coin 4 -> dp[1]+1=2. dp[5]=2
# dp[6]: try coin 1 -> dp[5]+1=3. try coin 3 -> dp[3]+1=2. try coin 4 -> dp[2]+1=3. dp[6]=2
# Answer: 2 (coins 3+3)
print(f"\nCoin change [1,3,4] amount=6: {coin_change([1, 3, 4], 6)}")  # 2
print(f"Coin change [1,5,10,25] amount=30: {coin_change([1, 5, 10, 25], 30)}")  # 2
print(f"Coin change [2] amount=3: {coin_change([2], 3)}")  # -1 (impossible)


# =============================================================================
# Problem 3: House Robber (LC 198)
# =============================================================================
# Cannot rob two adjacent houses. Maximize total money.
#
# State:   dp[i] = max money from first i houses
# Recurrence: dp[i] = max(dp[i-1], dp[i-2] + nums[i])
#             (skip house i, or rob house i and take best from 2 back)
# Base case: dp[0] = nums[0], dp[1] = max(nums[0], nums[1])
# Answer:  dp[n-1]

def rob(nums: list[int]) -> int:
    """O(n) time, O(1) space."""
    if not nums:
        return 0
    if len(nums) == 1:
        return nums[0]

    prev2 = nums[0]
    prev1 = max(nums[0], nums[1])

    for i in range(2, len(nums)):
        curr = max(prev1, prev2 + nums[i])
        prev2, prev1 = prev1, curr

    return prev1


# Trace for nums=[2, 7, 9, 3, 1]:
# dp[0]=2, dp[1]=max(2,7)=7
# dp[2] = max(dp[1], dp[0]+9) = max(7, 11) = 11
# dp[3] = max(dp[2], dp[1]+3) = max(11, 10) = 11 (skip house 3, not worth it)
# dp[4] = max(dp[3], dp[2]+1) = max(11, 12) = 12
# Answer: 12 (rob houses 0, 2, 4 -> 2+9+1=12)
print(f"\nHouse robber [2,7,9,3,1]: {rob([2, 7, 9, 3, 1])}")  # 12
print(f"House robber [1,2,3,1]: {rob([1, 2, 3, 1])}")         # 4
print(f"House robber [2,1,1,2]: {rob([2, 1, 1, 2])}")         # 4


# --- Top-down (memoized) version of coin_change for comparison ---
def coin_change_memo(coins: list[int], amount: int) -> int:
    """Same complexity, top-down style."""

    @cache
    def dp(remaining: int) -> int:
        if remaining == 0:
            return 0
        if remaining < 0:
            return float("inf")
        return min((dp(remaining - c) + 1 for c in coins), default=float("inf"))

    result = dp(amount)
    dp.cache_clear()
    return result if result != float("inf") else -1


print(f"\nCoin change (memo) [1,3,4] amount=6: {coin_change_memo([1, 3, 4], 6)}")

# Try next:
# 1. Solve Longest Common Subsequence (LC 1143) -- 2D DP with string comparison.
# 2. Optimize coin_change space to O(amount) (already is!) -- now try to reconstruct which coins.
# 3. Solve "unique paths" in a grid -- classic 2D DP.

## Drill Ladder

- implement a hash map or LRU-style structure from scratch
- solve one BFS/DFS problem and explain why graph traversal fits
- solve one topological sort problem and relate it to DAG scheduling
- solve one binary-search variant and explain boundary handling
- solve one sliding-window problem and explain state updates
- solve one DP problem and write the state and recurrence explicitly
- explain time and space complexity for every retained solution

## Problem Bank (from design3.md)

Target time indicates readiness. If a problem takes more than 2x target, the underlying pattern needs more drill.

| Pattern | Problem | Target |
|---------|---------|--------|
| Hash map | Two Sum (LC 1) | 5 min |
| Hash map | Group Anagrams (LC 49) | 10 min |
| Sliding window | Longest Substring Without Repeating Characters (LC 3) | 10 min |
| DFS | Number of Islands (LC 200) | 15 min |
| DFS | Max Depth of Binary Tree (LC 104) | 5 min |
| DFS | Word Search (LC 79) | 15 min |
| BFS | Binary Tree Level Order Traversal (LC 102) | 10 min |
| Topological sort | Course Schedule (LC 207) | 15 min |
| Binary search | Binary Search (LC 704) | 5 min |
| Binary search | Search in Rotated Sorted Array (LC 33) | 15 min |
| Stack | Valid Parentheses (LC 20) | 5 min |
| DP | Climbing Stairs (LC 70) | 5 min |
| DP | Coin Change (LC 322) | 15 min |
| DP | House Robber (LC 198) | 10 min |
| DP | Longest Common Subsequence (LC 1143) | 15 min |
| Heap | Find Median from Data Stream (LC 295) | 15 min |
| Heap | Merge Intervals (LC 56) | 10 min |

## Artifact Target

- a small `data-structure lab`
- notes explaining the structure choice for each problem


In [ ]:
from collections import deque


# --- BFS template ---
def bfs(graph, start):
    visited = {start}
    queue = deque([start])
    order = []
    while queue:
        node = queue.popleft()
        order.append(node)
        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
    return order


# --- DFS template (iterative) ---
def dfs(graph, start):
    visited = set()
    stack = [start]
    order = []
    while stack:
        node = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        order.append(node)
        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                stack.append(neighbor)
    return order


# --- Topological sort (Kahn's algorithm) ---
def topological_sort(num_nodes, edges):
    graph = {i: [] for i in range(num_nodes)}
    indegree = [0] * num_nodes
    for src, dst in edges:
        graph[src].append(dst)
        indegree[dst] += 1
    queue = deque([i for i in range(num_nodes) if indegree[i] == 0])
    order = []
    while queue:
        node = queue.popleft()
        order.append(node)
        for neighbor in graph[node]:
            indegree[neighbor] -= 1
            if indegree[neighbor] == 0:
                queue.append(neighbor)
    return order if len(order) == num_nodes else []


# --- Binary search template ---
def binary_search(nums, target):
    left, right = 0, len(nums) - 1
    while left <= right:
        mid = (left + right) // 2
        if nums[mid] == target:
            return mid
        if nums[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1


# --- Sliding window template ---
def longest_unique_substring_length(s):
    seen = {}
    left = 0
    best = 0
    for right, ch in enumerate(s):
        if ch in seen and seen[ch] >= left:
            left = seen[ch] + 1
        seen[ch] = right
        best = max(best, right - left + 1)
    return best


# --- DP pattern ---
# 1. Define state: dp[i] = answer for subproblem i
# 2. Find recurrence: dp[i] = f(dp[i-1], dp[i-2], ...)
# 3. Base case: dp[0] = ...
# 4. Build bottom-up (iterate) or top-down (memoize)
# 5. Answer = dp[n]


def climb_stairs(n: int) -> int:
    if n <= 2:
        return n
    dp = [0] * (n + 1)
    dp[1], dp[2] = 1, 2
    for i in range(3, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n]


## 11. Problem Bank

Target time indicates readiness. If a problem takes more than 2x target, the underlying pattern needs more drill.

| Pattern | Problem | Target | Key Insight |
|---------|---------|--------|-------------|
| Hash map | Two Sum (LC 1) | 5 min | Complement lookup in hash map |
| Hash map | Group Anagrams (LC 49) | 10 min | Canonical form as dict key |
| Sliding window | Longest Substring Without Repeating (LC 3) | 10 min | Expand/shrink window on repeat |
| DFS | Number of Islands (LC 200) | 15 min | Connected components on grid |
| DFS | Max Depth of Binary Tree (LC 104) | 5 min | Recursive depth = 1 + max(children) |
| DFS | Word Search (LC 79) | 15 min | Backtracking DFS on grid |
| BFS | Binary Tree Level Order Traversal (LC 102) | 10 min | Queue processes one level at a time |
| Topological sort | Course Schedule (LC 207) | 15 min | Cycle detection via Kahn's |
| Binary search | Binary Search (LC 704) | 5 min | Standard halving |
| Binary search | Search in Rotated Sorted Array (LC 33) | 15 min | One half always sorted |
| Stack | Valid Parentheses (LC 20) | 5 min | Push open, pop on close, check match |
| DP | Climbing Stairs (LC 70) | 5 min | dp[i] = dp[i-1] + dp[i-2] |
| DP | Coin Change (LC 322) | 15 min | dp[i] = min(dp[i-coin]+1) |
| DP | House Robber (LC 198) | 10 min | dp[i] = max(skip, rob+dp[i-2]) |
| DP | Longest Common Subsequence (LC 1143) | 15 min | 2D DP on two strings |
| Heap | Find Median from Data Stream (LC 295) | 15 min | Two heaps split at median |
| Heap | Merge Intervals (LC 56) | 10 min | Sort by start, merge overlaps |

## 12. Mini Lab

Solve these three problems from scratch. For each one:
1. State which data structure / pattern you are using and why
2. Write the solution
3. State time and space complexity
4. Test with at least 2 cases including an edge case

### Problem A: Graph -- Course Schedule II (LC 210)

Given `numCourses` and prerequisite pairs, return an ordering of courses you should take to finish all courses. If impossible, return an empty list. (This is topological sort with output.)

### Problem B: DP -- Longest Common Subsequence (LC 1143)

Given two strings `text1` and `text2`, return the length of their longest common subsequence. A subsequence is derived by deleting some (or no) characters without changing the order of remaining characters.

Hint: 2D DP. `dp[i][j]` = LCS length of `text1[:i]` and `text2[:j]`.

### Problem C: Hash Map -- Subarray Sum Equals K (LC 560)

Given an array of integers and an integer `k`, return the total number of subarrays whose sum equals `k`.

Hint: prefix sum + hash map counting how many times each prefix sum has occurred.

## 13. Exit Criteria

You are ready to move on only if you can do all of the following without searching:

**Complexity analysis**:
- State the time and space complexity of any solution you write
- Explain why `list.append` is O(1) amortized
- Explain why space complexity often matters more than time in data pipelines

**Data structure selection**:
- Given a problem, choose a data structure and justify the choice with complexity reasoning
- Explain when a hash map degrades from O(1) to O(n)
- Explain why `deque` is better than `list` for queue workloads

**Pattern recognition**:
- Identify a sliding window problem from problem description
- Identify when a problem is graph-shaped (dependencies, connectivity, reachability)
- Identify when a problem needs DP (overlapping subproblems, optimal substructure)
- Explain the difference between BFS and DFS and when to prefer each

**Implementation**:
- Implement BFS, DFS, topological sort, binary search, sliding window, and basic DP from memory
- Solve any problem in the Problem Bank within the target time
- Complete all three Mini Lab problems with correct complexity analysis

**Industry connection**:
- Relate topological sort to Airflow DAG scheduling
- Relate sliding window to streaming aggregation and rate limiting
- Relate heaps to priority task scheduling
- Relate hash maps to message deduplication and symbol lookups

## Interview Question Bank

Use these after you finish the notebook. Answer out loud, not just in writing.

- How do you decide whether a problem is hash-map-shaped, graph-shaped, sliding-window-shaped, or DP-shaped?
- Why is `list.append` often described as amortized O(1) rather than strictly O(1)?
- When would you choose BFS over DFS, and when would DFS be the better fit?
- Why is topological sort the right model for dependency scheduling problems?
- What are the common edge cases that break binary search implementations?
- Why does a heap help for top-K problems without sorting everything?
- What are the two signals that a problem may need dynamic programming?
- Given a solution, how do you justify the time and space complexity clearly in an interview?


## 14. References

- Python Time Complexity Wiki
  https://wiki.python.org/moin/TimeComplexity
- Python `collections` module
  https://docs.python.org/3/library/collections.html
- Python `heapq` module
  https://docs.python.org/3/library/heapq.html
- Python `bisect` module
  https://docs.python.org/3/library/bisect.html
- Python `functools.cache` (memoization)
  https://docs.python.org/3/library/functools.html#functools.cache
- Introduction to Algorithms (CLRS) -- Cormen, Leiserson, Rivest, Stein
  Chapters on sorting, graph algorithms, dynamic programming, and amortized analysis
- Airflow documentation on DAG scheduling
  https://airflow.apache.org/docs/apache-airflow/stable/core-concepts/dags.html